In [2]:
import os
import sys
import re
import time

project_root = "/root/work/tenset"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")
    

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build:" + os.environ.get("LD_LIBRARY_PATH", "")

1. Total 저장

In [3]:
from glob import glob
import pandas as pd


dir_name = "/root/work/tenset/scripts/pre_experiments/model_myself/result/(8c674f26f66543069d1e1c56cda249f9,4,60,60,256,1,1,256,512,1,1,1,512,4,30,30,512,cuda)"

csv_dir = glob(f"{dir_name}/**/*.csv", recursive=True)

csvs = []
for csv in csv_dir:
    if "avg" in csv or "total" in csv or "sampling" in csv or "prev" in csv:
       continue
    csvs.append(csv) 

dfs = []
for p in csvs:
    sub_df = pd.read_csv(p)
    sub_df = sub_df.loc[:, ~sub_df.columns.str.startswith("Unnamed")]
    if "rank_warmup_epochs" not in sub_df.columns:
        sub_df["rank_warmup_epochs"] = 0
    if "measure_size" not in sub_df.columns:
        sub_df["measure_size"] = 64
    if "scratch" not in sub_df.columns:
        sub_df["scratch"] = False
    if "encoder_freeze" not in sub_df.columns:
        sub_df["encoder_freeze"] = False
    if "lambda_pair" not in sub_df.columns:
        sub_df["lambda_pair"] = 3.0
    if "lambda_infonce" not in sub_df.columns:
        sub_df["lambda_infonce"] = 0.0
    if "tau" not in sub_df.columns:
        sub_df["tau"] = 0.0
    if "tau_c" not in sub_df.columns:
        sub_df["tau_c"] = None
    if "beta" not in sub_df.columns:
        sub_df["beta"] = 0.01
    if "gamma" not in sub_df.columns:
        sub_df["gamma"] = 0.01
    if "noise_std" not in sub_df.columns:
        sub_df["noise_std"] = 0.001
    if "margin_scale" not in sub_df.columns:
        sub_df["margin_scale"] = 0.3
    if "cost_predictor_lr" not in sub_df.columns:
        sub_df["cost_predictor_lr"] = 0.01
    
    # if "T_mc" not in sub_df.columns:
    #     sub_df["T_mc"] = 20
    
    dfs.append(sub_df)

    
df_total = pd.concat(dfs, ignore_index=True)

# measure_size 컬럼을 맨 앞으로 이동
cols = df_total.columns.tolist()
df_total = df_total[["measure_size"] + [c for c in cols if c != "measure_size"]]

# T_mc drop
df_total = df_total.drop(columns=["T_mc"], errors='ignore')
df_total = df_total.drop(columns=["lambda_smooth"], errors='ignore')

df_total.to_csv(f"{dir_name}/vae_extent_total.csv", index=True)
df_total

,measure_size,encoder_freeze,scratch,weights,uncertainty_topk,grad_num,rand_num,encoder_lr,cost_predictor_lr,rank_warmup_epochs,...,sampling_seed,lambda_infonce,tau,tau_c,beta,gamma,noise_std,lambda_reg,feature_predictor_lr,epochs
0,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,2000,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
1,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,2001,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
2,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,2002,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
3,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,2003,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
4,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,2004,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17952,48,False,False,"(0.4, 0.3, 0.3)",NaN,4,0,0.00001,0.00001,200,...,2007,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
17953,48,False,False,"(0.4, 0.3, 0.3)",NaN,4,0,0.00001,0.00001,200,...,2008,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
17954,64,False,False,"(0.4, 0.3, 0.3)",64.0,4,0,NaN,0.01000,0,...,2000,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN
17955,64,False,False,"(0.4, 0.3, 0.3)",64.0,4,0,NaN,0.01000,0,...,2001,0.0,0.0,None,0.01,0.01,0.001,NaN,NaN,NaN


2. Total avg 저장

In [4]:
agg_kwargs = {
    "phase": ("phase", "mean"),
    "train_size": ("train_size", "mean"),
    "used_time": ("used_time", "mean"),
    "val_reg_r2": ("val_reg_r2", "first"),
    "seed_n": ("sampling_seed", "nunique"),
    "sampling_seed": ("sampling_seed", list),
}

topk_col = f"top-1"
if topk_col in df_total.columns:
    agg_kwargs[topk_col] = (topk_col, "mean")

ignore_group_cols = {
    "phase",
    "train_size",
    "used_time",
    "top-1",
    "val_reg_r2",
    "val_rank_r2",
    "sampling_seed",  # seed는 집계 대상이지 그룹 기준 아님
}

group_cols = [
    c for c in df_total.columns
    if c not in ignore_group_cols
]


df_total_avg = (
    df_total
    .groupby(group_cols, as_index=False, dropna=False)
    .agg(**agg_kwargs)
)

tail_cols = ["used_time", "val_reg_r2", "seed_n", "sampling_seed"]

cols = list(df_total_avg.columns)
front_cols = [c for c in cols if c not in tail_cols]

df_total_avg = df_total_avg[front_cols + tail_cols]

# seed_n이 20이 아닌 행 필터링
df_total_avg = df_total_avg[df_total_avg["seed_n"] == 20]

df_total_avg.to_csv(f"{dir_name}/vae_extent_total_avg.csv", index=False)
df_total_avg

,measure_size,encoder_freeze,scratch,weights,uncertainty_topk,grad_num,rand_num,encoder_lr,cost_predictor_lr,rank_warmup_epochs,...,lambda_reg,feature_predictor_lr,epochs,phase,train_size,top-1,used_time,val_reg_r2,seed_n,sampling_seed
0,48,False,False,"(0.4, 0.3, 0.3)",NaN,2,0,0.00001,0.00001,0,...,NaN,NaN,NaN,3.966667,190.4,0.0,13.798417,"[0.6567, 0.5815, 0.5721, 0.5986]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
1,48,False,False,"(0.4, 0.3, 0.3)",NaN,2,0,0.00001,0.00001,100,...,NaN,NaN,NaN,4.133333,198.4,0.0,14.423083,"[0.6782, 0.5936, 0.625, 0.6378]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
2,48,False,False,"(0.4, 0.3, 0.3)",NaN,2,0,0.00001,0.00001,200,...,NaN,NaN,NaN,3.966667,190.4,0.0,13.923417,"[0.6805, 0.5926, 0.6503, 0.6583]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
3,48,False,False,"(0.4, 0.3, 0.3)",NaN,2,0,0.00001,0.00010,0,...,NaN,NaN,NaN,3.383333,162.4,0.0,11.675833,"[0.6385, 0.5992, 0.6772, 0.6377]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
4,48,False,False,"(0.4, 0.3, 0.3)",NaN,2,0,0.00001,0.00010,100,...,NaN,NaN,NaN,3.383333,162.4,0.0,11.714667,"[0.6311, 0.6611, 0.703, 0.6609]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,NaN,NaN,NaN,2.700000,172.8,0.0,9.387000,"[0.6052, 0.6311, 0.6198, 0.6371]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
177,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,0.01,0.0,1000.0,2.550000,163.2,0.0,11.001000,"[0.6023, 0.6178, 0.6353, 0.6186]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
178,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,0.01,0.0,1000.0,2.800000,179.2,0.0,12.170000,"[0.5846, 0.6082, 0.6101, 0.6497]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
179,64,False,False,"(0.6, 0.1, 0.3)",64.0,4,0,0.00001,0.00010,100,...,0.01,0.0,1000.0,3.100000,198.4,0.0,13.753000,"[0.5736, 0.5873, 0.6039, 0.638]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."


In [ ]:
agg_kwargs = {
    "phase": ("phase", "mean"),
    "train_size": ("train_size", "mean"),
    "used_time": ("used_time", "mean"),
    "val_reg_r2": ("val_reg_r2", "first"),
    "seed_n": ("sampling_seed", "nunique"),
    "sampling_seed": ("sampling_seed", list),
}

topk_col = f"top-1"
if topk_col in df_total.columns:
    agg_kwargs[topk_col] = (topk_col, "mean")

group_cols = [
    "measure_size",
    "scratch",
    "encoder_freeze",
    "encoder_lr",
    "cost_predictor_lr",
    "rank_warmup_epochs",
    "lambda_pair",
    "weights",
    "uncertainty_topk",
    "grad_num",
    "rand_num",
]

df_total_avg = (
    df_total
    .groupby(group_cols, as_index=False, dropna=False)
    .agg(**agg_kwargs)
)

# 

df_total_avg.to_csv("/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda)/vae_extent_total_avg.csv", index=False)
df_total_avg

,measure_size,scratch,encoder_freeze,encoder_lr,cost_predictor_lr,rank_warmup_epochs,lambda_pair,weights,uncertainty_topk,grad_num,rand_num,phase,train_size,used_time,val_reg_r2,seed_n,sampling_seed,top-1
0,32,False,False,0.00001,0.00001,100,3.0,"(0.4, 0.3, 0.3)",0,2,0,6.75,230.4,44.547500,[0.4545],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.150000
1,32,False,False,0.00001,0.00001,100,3.0,"(0.4, 0.3, 0.3)",0,4,0,5.80,201.6,31.561500,[0.4545],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.200000
2,32,False,False,0.00001,0.00001,100,3.0,"(0.4, 0.3, 0.3)",64,2,0,4.00,144.0,17.559444,"[0.3615, 0.1561, 0.1386, 0.1755, 0.2463, 0.257...",18,"[2002, 2003, 2004, 2005, 2006, 2007, 2008, 200...",0.166667
3,32,False,False,0.00001,0.00001,100,3.0,"(0.4, 0.3, 0.3)",64,4,0,4.55,161.6,23.844500,[0.4545],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.250000
4,32,False,False,0.00001,0.00001,200,3.0,"(0.4, 0.3, 0.3)",0,2,0,5.50,192.0,29.047500,[0.4735],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,64,True,False,0.00001,0.00010,200,3.0,"(0.5, 0.2, 0.3)",64,4,0,2.20,140.8,7.565500,"[0.7547, 0.6704, 0.6467]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.100000
818,64,True,False,0.00001,0.00010,200,3.0,"(0.6, 0.1, 0.3)",64,2,0,2.55,163.2,8.957000,"[0.7547, 0.6943, 0.613]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.000000
819,64,True,False,0.00001,0.00010,200,3.0,"(0.6, 0.1, 0.3)",64,4,0,2.20,140.8,7.591000,"[0.7547, 0.6884]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.050000
820,64,True,False,0.00001,0.00010,200,3.0,"(0.7, 0.3, 0.0)",64,2,0,3.10,198.4,10.872000,"[0.7547, 0.6507, 0.6153]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.000000


선택 파일 Avg 저장

In [6]:
import pandas as pd
def save_result_avg(filename, top_k):
    df_results = pd.read_csv(filename)
    df_results = df_results.loc[:, ~df_results.columns.str.startswith("Unnamed")]
    ignore_cols = {
        "phase",
        "train_size",
        "used_time",
        f"top-{top_k}",
        "val_reg_r2",
        "val_rank_r2",
        "sampling_seed",  # seed는 그룹키가 아니라 집계 대상
    }

    group_cols = [c for c in df_results.columns if c not in ignore_cols]

    agg_dict = dict(
        phase=("phase", "mean"),
        train_size=("train_size", "mean"),
        used_time=("used_time", "mean"),
        val_reg_r2=("val_reg_r2", "first"),
        val_rank_r2=("val_rank_r2", "first"),
        seed_n=("sampling_seed", "nunique"),
        sampling_seed=("sampling_seed", list),
    )

    topk_col = f"top-{top_k}"
    if topk_col in df_results.columns:
        agg_dict[topk_col] = (topk_col, "mean")

    df_avg = (
        df_results
        .groupby(group_cols, as_index=False, dropna=False)
        .agg(**agg_dict)
    )

    # used_time, val_reg_r2, val_rank_r2, seed_n, sampling_seed를 맨 뒤로
    tail_cols = ["used_time", "val_reg_r2", "val_rank_r2", "seed_n", "sampling_seed"]
    tail_cols = [c for c in tail_cols if c in df_avg.columns]
    front_cols = [c for c in df_avg.columns if c not in tail_cols]
    df_avg = df_avg[front_cols + tail_cols]

    df_avg.to_csv(filename.replace(".csv", "_avg.csv"), index=False)
    return df_avg


df_avg = save_result_avg("/root/work/tenset/scripts/pre_experiments/model_myself/result/(0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960,cuda)/vae_extent_1229_2008.csv", 1)

df_avg

,encoder_freeze,scratch,measure_size,weights,uncertainty_topk,grad_num,rand_num,T_mc,encoder_lr,cost_predictor_lr,...,tau,tau_c,phase,train_size,top-1,used_time,val_reg_r2,val_rank_r2,seed_n,sampling_seed
0,False,False,64,"(0.4, 0.3, 0.3)",64,4,0,20,0.00001,0.0001,...,0.1,NaN,1.40,89.6,0.350000,5.865500,[0.688],[None],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
1,False,False,64,"(0.4, 0.3, 0.3)",64,4,0,20,0.00001,0.0001,...,0.1,NaN,1.55,99.2,0.350000,6.444000,[0.6877],[None],20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
2,False,False,64,"(0.4, 0.3, 0.3)",64,4,0,20,0.00001,0.0001,...,0.1,NaN,1.00,64.0,0.333333,4.166667,[0.6848],[None],3,"[2000, 2001, 2002]"


두 csv 합치기

In [ ]:
import pandas as pd


# CSV 파일 읽기
csv1 = pd.read_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2013.csv')
csv2 = pd.read_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv')

# 두 CSV 합치기
merged_csv = pd.concat([csv1, csv2])

# 합친 CSV 저장
merged_csv.to_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_hyper_total.csv', index=False)
save_avg_csv(merged_csv, '/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_hyper_total.csv', top_k=1)

In [13]:
h = pd.read_csv("/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv")

h["train_size"] = h["train_size"] + 16
h.to_csv(
    "/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv",
    index=False
)
